# AQUAVOLTA — Cleaning of collections

Sheet `ENCAISSEMENTS` of `DATA BI.xlsx`.

**How to run** — install the dependencies once (`pip install -r analytics/requirements.txt`), place the confidential workbook in `data/raw/` (or set `AQUAVOLTA_DATA_DIR`), then run all cells from top to bottom. Each section reloads its own sheet into `res`, so sections can also be run independently after the two setup cells.

The source data is confidential: this notebook is published without outputs. See `analytics/README.md` and `analytics/calculation-audit.md`.

## Setup

In [ ]:
import pandas as pd
import missingno as msno
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

In [ ]:
# Paths and loading helpers (see aquavolta_helpers.py)
from aquavolta_helpers import DATA_DIR, OUTPUT_DIR, load_sheet, save_output

print("Source workbooks :", DATA_DIR)
print("Cleaned outputs  :", OUTPUT_DIR)

## Sheet `ENCAISSEMENTS`

Source: `DATA BI.xlsx` (header on row 3).

In [ ]:
res = load_sheet("DATA BI.xlsx", sheet_name='ENCAISSEMENTS', header=2)
print(res)

In [ ]:
print(res.shape)

In [ ]:
print(res.info())

In [ ]:
print(res.describe())

In [ ]:
msno.bar(res)

In [ ]:
mode=res[["BANQUE"]].mode().iloc[0]
for coll in mode.index:
    res[coll]=res[coll].fillna(mode[coll])
print(res.info())

> **Review note (audit finding F4).** `paiement TND` and `TCL` are imputed by a regression model. A missing payment may simply mean that the invoice has not been paid yet; an estimated payment then creates cash that was never received. Calculation left unchanged; imputed rows should be flagged or excluded from cash KPIs until confirmed.

In [ ]:
# 2. Kol chay bel numeric (on focus 3la colonnes importantes)
cols_numeric = ["CA €", "tx conv", "CA TND", "paiement TND", "TCL"]
data_num = res[cols_numeric]

# 3. Itérative Imputer
imputer = IterativeImputer(estimator=BayesianRidge(), random_state=0)
data_imputed = imputer.fit_transform(data_num)

# 4. Remplacer les anciennes colonnes bel nouvelles
res[cols_numeric] = data_imputed
print(res.info())

In [ ]:
print(res)

In [ ]:
res["RECU (frais déduits)"] = res["RECU (frais déduits)"].apply(
    lambda x: "PAYE" if isinstance(x, (int, float)) else x)
print(res)

In [ ]:
res["DATE"] = pd.to_datetime(res["DATE"]).dt.strftime("%d/%m/%Y")
print(res)

In [ ]:
print(res["DATE"])

In [ ]:
def clean_date(value):
    try:
       
        if pd.isna(value):
            return pd.NaT

       
        value = str(value).strip()

        parts = value.split("/")
        if len(parts) == 3 and len(parts[2]) < 4:
            
            year = parts[2].zfill(4)  
            if int(year) < 1900:      
                year = "20" + year[-2:]
            value = f"{parts[0]}/{parts[1]}/{year}"

       
        return pd.to_datetime(value, dayfirst=True, errors="coerce")

    except Exception:
        return pd.NaT


res["DATE ENC"] = res["DATE ENC"].apply(clean_date)

res["DATE ENC"] = res["DATE ENC"].dt.strftime("%d/%m/%Y")
print(res)

In [ ]:
save_output(res, "output____ENCAISSEMENT.xlsx")